# Love Island Manual Downloads
Because the Reddit API is closed to researchers at an institution, I had to get some of the larger Love Island threads from manual download. While this process was easy, it was rather expensive.

- Do not use Apify, though it is more complete.
- Use Export Comments, because the data is the text anyway. 

In [1]:
import os

import re              # EXTRACT SUBMISSION IDS
import subprocess      # RUN BDFR
import time            # TRACK TIME AND SLEEP
import pandas as pd

from openpyxl import load_workbook

from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

import duckdb


In [2]:
# SET PATHS
data_dir   = Path("../data/LoveIslandUSA_S8_Manual/")
xlsx_files = list(data_dir.glob("*.xlsx"))

print(
    f"N XLSX FILES: {len(xlsx_files)}"
)

N XLSX FILES: 99


## Reading Excel Files

### Function: Read ID's from Excel File

In [3]:
import re
import pandas as pd
from openpyxl import load_workbook

def get_ids_from_xlsx(file_path):
    skiprows = 6
    header_excel_row = skiprows + 1
    data_start_row = header_excel_row + 1

    # Must match the settings in read_reddit_thread()
    tmp = pd.read_excel(
        file_path,
        skiprows=skiprows,
        header=0,
    )

    wb = load_workbook(file_path, data_only=True)
    ws = wb.active

    headers = {
        cell.value: cell.column
        for cell in ws[header_excel_row]
    }

    url_col = headers["(view source)"]

    url_list = []

    for row in range(data_start_row, data_start_row + len(tmp)):
        cell = ws.cell(row=row, column=url_col)

        url = (
            cell.hyperlink.target
            if cell.hyperlink
            else cell.value
        )

        url_list.append(url)

    submission_ids = [
        (
            re.search(r"/comments/([^/]+)", url).group(1)
            if url and re.search(r"/comments/([^/]+)", url)
            else None
        )
        for url in url_list
    ]

    comment_ids = [
        url.rstrip("/").split("/")[-1]
        if url
        else None
        for url in url_list
    ]

    return submission_ids, comment_ids, url_list

### Function: Read Comments from Excel Files

In [4]:
def read_reddit_thread(this_xlsx):
    submission_ids, comment_ids, url_list = get_ids_from_xlsx(this_xlsx)

    tmp = pd.read_excel(
        this_xlsx,
        skiprows=6,
        header=0,
    )

    tmp = tmp.rename(columns={
        tmp.columns[0]: "comment_number",
        tmp.columns[1]: "subcomment_number",
        tmp.columns[2]: "author",
    })

    tmp["post_id"] = submission_ids
    tmp["id"] = comment_ids
    tmp["url"] = url_list

    return tmp

## DF: Comments DataFrame

In [5]:
frames = []
errors = []

for file in tqdm(xlsx_files, desc="Reading Excel files"):
    try:
        tmp = read_reddit_thread(file)
        tmp["source_file"] = Path(file).name
        frames.append(tmp)
    except Exception as error:
        errors.append((file, str(error)))

Reading Excel files:   0%|          | 0/99 [00:00<?, ?it/s]

In [6]:
comments_df = pd.concat(frames, ignore_index=True)

In [7]:
# Check the results
print(comments_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194086 entries, 0 to 194085
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   comment_number     170240 non-null  float64       
 1   subcomment_number  23846 non-null   object        
 2   author             192640 non-null  object        
 3   Date               194086 non-null  datetime64[ns]
 4   Score              194086 non-null  int64         
 5   Message            192640 non-null  object        
 6   isPremium          194086 non-null  object        
 7   (view source)      194086 non-null  object        
 8   post_id            194086 non-null  object        
 9   id                 194086 non-null  object        
 10  url                194086 non-null  object        
 11  source_file        194086 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(9)
memory usage: 17.8+ MB
None


### Connect Submission Text/Authors
The following object holds the names of the text files that do not have corresponding JSON files, i.e. the `bdfr` request failed.

Thus, we will need to look into their .txt files for this information:
- Author,  
- Title,  
- Post ID (which we will use to connect to the `comments_df`)

In [8]:
from pathlib import Path

folder = Path('../data/LoveIslandUSA_S8/')

txt_stems = {file.stem for file in folder.glob("*.txt")}
json_stems = {file.stem for file in folder.glob("*.json")}

txt_without_json = sorted(txt_stems - json_stems)

txt_without_json[0:3]

['AutoModerator_Season 8 - Episode 1 - Cast Opinions Discussion_1tvbdpi',
 'AutoModerator_Season 8 - Episode 1 - Post Episode Discussion_1tvbdoe',
 'AutoModerator_Season 8 - Episode 12 - Cast Opinions Discussion_1u6zpx2']

In [9]:
from pathlib import Path
import re

def get_info_from_txt(txt_stem, folder):
    """
    Extract post author, title, post ID, and submission body from a .txt file.
    """
    # Ensure we have only the filename stem
    txt_stem = Path(txt_stem).stem

    post_author = txt_stem.split("_", 1)[0]
    title = txt_stem.split("_", 1)[1].rsplit("_", 1)[0]
    post_id = txt_stem.rsplit("_", 1)[-1]

    file_path = Path(folder) / f"{txt_stem}.txt"
    text = file_path.read_text(encoding="utf-8")

    match = re.search(
        r"(^#[^#].*?)^---\s*$",
        text,
        flags=re.MULTILINE | re.DOTALL,
    )

    submission_body = match.group(1).strip() if match else None

    return post_author, title, post_id, submission_body

In [10]:
texts = []
errors = []

for text_stem in tqdm(txt_without_json, desc="Reading text files"):
    try:
        tmp = get_info_from_txt(text_stem, folder)
        texts.append(tmp)
    except Exception as error:
        errors.append((text_stem, str(error)))

post_info_df = pd.DataFrame(texts, columns=["post_author", "title", "post_id", 'submission_body'])

Reading text files:   0%|          | 0/102 [00:00<?, ?it/s]

In [11]:
post_info_df.head()

,post_author,title,post_id,submission_body
0,AutoModerator,Season 8 - Episode 1 - Cast Opinions Discussion,1tvbdpi,# Let's discuss!\n\nShare your opinions on the...
1,AutoModerator,Season 8 - Episode 1 - Post Episode Discussion,1tvbdoe,# Let's discuss!\n\nCan't find a megathread? U...
2,AutoModerator,Season 8 - Episode 12 - Cast Opinions Discussion,1u6zpx2,# Let's discuss!\n\nShare your opinions on the...
3,AutoModerator,Season 8 - Episode 12 - Post Episode Discussion,1u6zpwm,# Let's discuss!\n\nCan't find a megathread? U...
4,AutoModerator,Season 8 - Episode 13 - Post Episode Discussion,1u7wfk9,# Let's discuss!\n\nCan't find a megathread? U...


Let's join this to the `comments_df`. This will help clarify whether we have all the data to move forward on these posts.

In [12]:
comments_df = comments_df.merge(
    post_info_df,
    on = 'post_id',
    how='left'
)

In [13]:
comments_df.head(3)

,comment_number,subcomment_number,author,Date,Score,Message,isPremium,(view source),post_id,id,url,source_file,post_author,title,submission_body
0,1.0,NaN,Holiday_Change4329,2026-07-10 01:29:21,69,anne in the confessionals actually had me roll...,no,view comment,1us9jhs,owm2bub,https://www.reddit.com/r/LoveIslandUSA/comment...,reddit-comments_season_8_episode_32_post_episo...,schedulerplus,Season 8 - Episode 32 - Post Episode Discussion,# Let's discuss!\n\nCan't find a megathread? U...
1,2.0,NaN,Sung__Jin-W00,2026-07-10 01:30:39,216,Anne would be a better aftersun host,no,view comment,1us9jhs,owm2ke5,https://www.reddit.com/r/LoveIslandUSA/comment...,reddit-comments_season_8_episode_32_post_episo...,schedulerplus,Season 8 - Episode 32 - Post Episode Discussion,# Let's discuss!\n\nCan't find a megathread? U...
2,3.0,NaN,Kooky-Technician8098,2026-07-10 01:30:52,8,You can lead a horse to water..,no,view comment,1us9jhs,owm2lqz,https://www.reddit.com/r/LoveIslandUSA/comment...,reddit-comments_season_8_episode_32_post_episo...,schedulerplus,Season 8 - Episode 32 - Post Episode Discussion,# Let's discuss!\n\nCan't find a megathread? U...


All comments are matched to their `post_id`.

In [14]:
anti_join_comments = (
    comments_df.merge(
        post_info_df[["post_id"]].drop_duplicates(),
        on=["post_id"],
        how="left",
        indicator=True,
    )
    .query('_merge == "left_only"')
    .drop(columns="_merge")
)

In [15]:
anti_join_comments

,comment_number,subcomment_number,author,Date,Score,Message,isPremium,(view source),post_id,id,url,source_file,post_author,title,submission_body


I would like to know how I can bind these rows to existing comments table in our database.

In [16]:
# 1. Connect to the existing database
con = duckdb.connect("../data/love_island_reddit.duckdb")

# 2. See preview of the comments DataFrame
con.execute("SELECT * FROM comments LIMIT 3").df()

,author,id,score,subreddit,author_flair,submission,stickied,body,is_submitter,distinguished,created_utc,parent_id,replies,depth,replied_to_top,post_title,post_id,post_score,source_file,created_at
0,AutoModerator,oqr6sxe,1,LoveIslandUSA,None,1u1o6bb,True,### **SORT** this Discussion &#x1F3A8;\n---\n-...,True,moderator,1.781055e+09,t3_1u1o6bb,<NA>,1,False,Season 8 - Episode 7 - Cast Opinions Discussion,1u1o6bb,14,../data/LoveIslandUSA_S8/AutoModerator_Season ...,2026-06-09 18:30:28
1,Lazy_Chemistry_3899,oqrjrnv,153,LoveIslandUSA,None,1u1o6bb,False,When Bryce bragged to the guys how kind and ni...,False,None,1.781059e+09,t3_1u1o6bb,<NA>,1,False,Season 8 - Episode 7 - Cast Opinions Discussion,1u1o6bb,14,../data/LoveIslandUSA_S8/AutoModerator_Season ...,2026-06-09 19:44:08
2,katieofgilead,oqrmj7f,35,LoveIslandUSA,None,1u1o6bb,False,Loooved that! He is looking way past her beaut...,False,None,1.781060e+09,oqrjrnv,<NA>,2,True,Season 8 - Episode 7 - Cast Opinions Discussion,1u1o6bb,14,../data/LoveIslandUSA_S8/AutoModerator_Season ...,2026-06-09 20:00:32


In [17]:
# 3. Close
con.close()

We have:
- `author = author`
- `source_file = source_file`

We can rename:  
- `submission = post_id = post_id (manual)`
- `body = submission_body`
- `post_title = title`
- `created_utc = ` the utc value of `Date`
- `post_score = Score`

We can create:
- `subreddit = 'LoveIslandUSA'`
- `replied_to_top = ` if `subcomment_number` is `NaN`
- `is_submitter = ` if `author` is `post_author` 
- `parent_id = ` the `comment_number` above the thread
- `depth = ` the `subcomment_number` + 1 (0 if `NaN`)


However, we do not have these data:

- `id = NULL`, which is the author/user ID
- `score = NULL`, which I assume is the author/user score
- `author_flair = NULL`, also about the author
- `post_score = NULL` 
- `stickied = NULL`
- `distinguished = NULL`


In [18]:
# We have a duplicated column, and I'd like to just keep it the same.
comments_df["submission"] = comments_df["post_id"]

# Renames
comments_df = comments_df.rename(columns={
    "Message": "body",
    "title": "post_title",
    "Date": "created_utc",
    "Score": "score"
})

# Create new columns
comments_df["subreddit"] = 'LoveIslandUSA'
comments_df['replied_to_top'] = comments_df["subcomment_number"].isna()
comments_df['is_submitter'] = comments_df['author'] == comments_df['post_author']
comments_df["depth"] = (
    comments_df["subcomment_number"]
    .notna()
    .astype(int)
)

# from datetime import datetime, UTC
# utc_now = datetime.now(UTC)
# comments_df['created_at'] = utc_now

In [19]:
# Add NA Columns
missing_cols = [
    "author_id",
    "author_score",
    "author_flair",
    "stickied",
    "distinguished",
    "post_score",
    "replies",
    "created_at"
]

comments_df[missing_cols] = pd.NA

In [20]:
import pandas as pd

def assign_parent_id(df):
    """
    Reconstruct parent_id using ExportComments numbering.

    Rules:
    - Top-level comments:
        parent_id = post_id
    - Replies such as "10-1":
        parent_id = the id of the row where comment_number == 10

    Assumes replies are encoded as first-level subcomments.
    """

    df = df.copy()

    # Normalize comment_number so values like 10 and 10.0 match reliably.
    comment_numbers = pd.to_numeric(
        df["comment_number"],
        errors="coerce"
    ).astype("Int64")

    # Lookup: comment_number -> Reddit comment id
    comment_lookup = dict(
        zip(
            comment_numbers[df["comment_number"].notna()],
            df.loc[df["comment_number"].notna(), "id"]
        )
    )

    def find_parent(row):
        subcomment_number = row["subcomment_number"]

        # Top-level comment: parent is the Reddit post/submission.
        if pd.isna(subcomment_number):
            return row["post_id"]

        # Example: "10-1" -> 10
        try:
            parent_comment_num = int(
                str(subcomment_number).split("-")[0]
            )
        except (ValueError, TypeError):
            return pd.NA

        return comment_lookup.get(parent_comment_num, pd.NA)

    df["parent_id"] = df.apply(find_parent, axis=1)

    return df

In [21]:
comments_df = assign_parent_id(comments_df)

In [22]:
comments_df["created_utc"] = pd.to_datetime(
    comments_df["created_utc"],
    unit="s",
    utc=True
)

In [23]:
comments_df.head(3)

,comment_number,subcomment_number,author,created_utc,score,body,isPremium,(view source),post_id,id,...,depth,author_id,author_score,author_flair,stickied,distinguished,post_score,replies,created_at,parent_id
0,1.0,NaN,Holiday_Change4329,2026-07-10 01:29:21+00:00,69,anne in the confessionals actually had me roll...,no,view comment,1us9jhs,owm2bub,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1us9jhs
1,2.0,NaN,Sung__Jin-W00,2026-07-10 01:30:39+00:00,216,Anne would be a better aftersun host,no,view comment,1us9jhs,owm2ke5,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1us9jhs
2,3.0,NaN,Kooky-Technician8098,2026-07-10 01:30:52+00:00,8,You can lead a horse to water..,no,view comment,1us9jhs,owm2lqz,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1us9jhs


In [24]:
con = duckdb.connect("../data/love_island_reddit.duckdb")


In [25]:
# Get the existing DuckDB table columns in order
table_cols = (
    con.execute("PRAGMA table_info('comments')")
       .df()["name"]
       .tolist()
)

table_cols

['author',
 'id',
 'score',
 'subreddit',
 'author_flair',
 'submission',
 'stickied',
 'body',
 'is_submitter',
 'distinguished',
 'created_utc',
 'parent_id',
 'replies',
 'depth',
 'replied_to_top',
 'post_title',
 'post_id',
 'post_score',
 'source_file',
 'created_at']

In [26]:
extra_in_df = [col for col in comments_df.columns if col not in table_cols]
missing_from_df = [col for col in table_cols if col not in comments_df.columns]

print("Extra DataFrame columns:", extra_in_df)
print("Missing DataFrame columns:", missing_from_df)

Extra DataFrame columns: ['comment_number', 'subcomment_number', 'isPremium', '(view source)', 'url', 'post_author', 'submission_body', 'author_id', 'author_score']
Missing DataFrame columns: []


In [27]:
table_cols = (
    con.execute("PRAGMA table_info('comments')")
      .df()["name"]
      .tolist()
)

In [28]:
comments_to_append = comments_df[table_cols].copy()

# Keep created_utc as numeric Unix seconds
comments_to_append["created_utc"] = pd.to_numeric(
    comments_to_append["created_utc"],
    errors="coerce"
)

# Create readable UTC datetime for DuckDB TIMESTAMP column
comments_to_append["created_at"] = (
    pd.to_datetime(
        comments_to_append["created_utc"],
        unit="s",
        utc=True,
        errors="coerce"
    )
    .dt.tz_localize(None)
)

In [29]:
con.register("new_comments", comments_to_append)

con.execute("""
    INSERT INTO comments
    SELECT n.*
    FROM new_comments AS n
    WHERE NOT EXISTS (
        SELECT 1
        FROM comments AS c
        WHERE c.id = n.id
    );
    """)

In [30]:
# con.execute("""
#     DESCRIBE comments
# """).df()

In [31]:
n_comments = con.execute("""
SELECT COUNT(*)
FROM comments
""").fetchone()[0]

print(f"{n_comments:,}")

205,306


In [32]:
con.close()

In [33]:
# DuckDB sanity checks for the comments table
con = duckdb.connect("../data/love_island_reddit.duckdb")


checks = con.execute("""
WITH summary AS (
    SELECT
        COUNT(*) AS total_comments,
        COUNT(DISTINCT id) AS unique_comment_ids,
        COUNT(*) - COUNT(DISTINCT id) AS duplicate_comment_rows,
        COUNT_IF(parent_id IS NULL) AS missing_parent_ids,
        COUNT_IF(created_utc IS NULL) AS missing_created_utc,
        COUNT_IF(created_at IS NULL) AS missing_created_at
    FROM comments
),

depth_counts AS (
    SELECT
        depth,
        COUNT(*) AS n_comments
    FROM comments
    GROUP BY depth
),

parent_type_counts AS (
    SELECT
        CASE
            WHEN parent_id = post_id THEN 'top-level comment'
            WHEN parent_id IS NULL THEN 'missing parent'
            ELSE 'reply to comment'
        END AS comment_type,
        COUNT(*) AS n_comments
    FROM comments
    GROUP BY 1
)

SELECT
    'summary' AS check_type,
    'total_comments' AS metric,
    CAST(total_comments AS VARCHAR) AS value
FROM summary

UNION ALL

SELECT
    'summary',
    'unique_comment_ids',
    CAST(unique_comment_ids AS VARCHAR)
FROM summary

UNION ALL

SELECT
    'summary',
    'duplicate_comment_rows',
    CAST(duplicate_comment_rows AS VARCHAR)
FROM summary

UNION ALL

SELECT
    'summary',
    'missing_parent_ids',
    CAST(missing_parent_ids AS VARCHAR)
FROM summary

UNION ALL

SELECT
    'summary',
    'missing_created_utc',
    CAST(missing_created_utc AS VARCHAR)
FROM summary

UNION ALL

SELECT
    'summary',
    'missing_created_at',
    CAST(missing_created_at AS VARCHAR)
FROM summary

UNION ALL

SELECT
    'depth',
    'depth_' || COALESCE(CAST(depth AS VARCHAR), 'NULL'),
    CAST(n_comments AS VARCHAR)
FROM depth_counts

UNION ALL

SELECT
    'comment_type',
    comment_type,
    CAST(n_comments AS VARCHAR)
FROM parent_type_counts

ORDER BY check_type, metric
""").df()

checks

,check_type,metric,value
0,comment_type,reply to comment,35066
1,comment_type,top-level comment,170240
2,depth,depth_0,170240
3,depth,depth_1,28032
4,depth,depth_10,8
5,depth,depth_11,4
6,depth,depth_12,3
7,depth,depth_13,1
8,depth,depth_14,1
9,depth,depth_2,4205


## DF: Submisisons DataFrame
1. Compare the column names between each.
2. Create the columns that are missing. 
3. Remove the ones that don't need to be there.
4. Append to existing database.

In [34]:
con = duckdb.connect("../data/love_island_reddit.duckdb")

con.execute("SELECT * FROM submissions LIMIT 3").df()

,title,name,url,selftext,score,upvote_ratio,permalink,id,author,link_flair_text,num_comments,over_18,spoiler,pinned,locked,distinguished,created_utc,created_at
0,Season 8 - Episode 7 - Cast Opinions Discussion,t3_1u1o6bb,https://www.reddit.com/r/LoveIslandUSA/comment...,# Let's discuss!\n\nShare your opinions on the...,14,0.85,/r/LoveIslandUSA/comments/1u1o6bb/season_8_epi...,1u1o6bb,AutoModerator,✴️ OPINIONS MEGATHREAD ✴️,374,False,False,False,False,None,1781055028,2026-06-09 18:30:28
1,[WITH Ads] Season 8 - Episode 10 -| 9 PM EST,t3_1u570dj,https://www.reddit.com/r/LoveIslandUSA/comment...,# Let's chat!\n\nThe with-ads live viewing sta...,6,0.88,/r/LoveIslandUSA/comments/1u570dj/with_ads_sea...,1u570dj,schedulerplus,LIVE DISCUSSION,55,False,False,False,False,moderator,1781398145,2026-06-13 17:49:05
2,[NONSPOILER] Season 8 - Episode 32 - Thursday ...,t3_1us8brl,https://www.reddit.com/r/LoveIslandUSA/comment...,# Let's chat!\n\nThe live discussion **spoiler...,9,0.84,/r/LoveIslandUSA/comments/1us8brl/nonspoiler_s...,1us8brl,schedulerplus,LIVE DISCUSSION | 😇 NONSPOILER 😇,220,False,False,False,False,moderator,1783643645,2026-07-09 17:34:05


Since we do not have created UTC's for the posts, we will just assume that posts were made specifically for the episode and came out on the day of the episode.

In [ ]:
posts_df = (
    comments_df[
        ["post_author", "post_title", "post_id", "submission_body"]
    ]
    .drop_duplicates(subset=["post_id"])
    .reset_index(drop=True)
)

posts_df["url"] = (
    "https://www.reddit.com/comments/"
    + posts_df["post_id"]
    + "/"
)

posts_df.head(5)

# FINISH THIS LATER:
# we dont have:
# score, upvote ratio, permalink, link_flair_text,
# spoiler, pinned, locked, distinguished,
# created_at, created_utc

# we can calculate number of comments


,post_author,post_title,post_id,submission_body,url
0,schedulerplus,Season 8 - Episode 32 - Post Episode Discussion,1us9jhs,# Let's discuss!\n\nCan't find a megathread? U...,https://www.reddit.com/comments/1us9jhs/
1,schedulerplus,Season 8 - Episode 20 - Post Episode Discussion,1ufss36,# Let's discuss!\n\nCan't find a megathread? U...,https://www.reddit.com/comments/1ufss36/
2,schedulerplus,[WITHOUT Ads] Season 8 - Finale - Sunday July ...,1uuwsvl,# Let's chat!\n\nThe **without-ads** live view...,https://www.reddit.com/comments/1uuwsvl/
3,schedulerplus,[WITHOUT Ads] Season 8 - Episode 24 - Monday J...,1ujaezn,# Let's chat!\n\nThe **without-ads** live view...,https://www.reddit.com/comments/1ujaezn/
4,schedulerplus,Season 8 - Episode 19 - Cast Opinions Discussion,1udzqzs,# Let's discuss!\n\nShare your opinions on the...,https://www.reddit.com/comments/1udzqzs/


In [36]:
comments_df.columns

Index(['comment_number', 'subcomment_number', 'author', 'created_utc', 'score',
       'body', 'isPremium', '(view source)', 'post_id', 'id', 'url',
       'source_file', 'post_author', 'post_title', 'submission_body',
       'submission', 'subreddit', 'replied_to_top', 'is_submitter', 'depth',
       'author_id', 'author_score', 'author_flair', 'stickied',
       'distinguished', 'post_score', 'replies', 'created_at', 'parent_id'],
      dtype='object')

Double check the above then save to duckdb.

FINISH THIS LATER:
We dont have:
- score, upvote ratio, permalink, link_flair_text,
- spoiler, pinned, locked, distinguished,
- created_at, created_utc

We can calculate number of comments.

## DF: Users DataFrame

In [37]:
con = duckdb.connect("../data/love_island_reddit.duckdb")

con.execute("SELECT * FROM users LIMIT 3").df()

,author,bot,num_submissions,num_comments
0,katieofgilead,N,0,3
1,lurkerchickk,N,0,1
2,DELETED,N,0,243


In [51]:
users_df_pt1 = (
    comments_df
    .groupby("author", dropna=False)["id"]
    .nunique()
    .reset_index(name="comment_count")
    .sort_values("comment_count", ascending=False)
    .reset_index(drop=True)
)

In [52]:
users_df_pt1

,author,comment_count
0,Silent_Advantage6138,2805
1,NaN,1446
2,TheOneThatCameEasy,1197
3,Lavendermin,1135
4,lilly_1005_2007,955
...,...,...
14352,Skylxrjane,1
14353,SkiUMah23,1
14354,Skeptical_optomist,1
14355,Skatedonthate999,1


In [53]:
users_df_pt2 = (
    comments_df
    .groupby("post_author", dropna=False)["post_id"]
    .nunique()
    .reset_index(name="post_count")
    .sort_values("post_count", ascending=False)
    .reset_index(drop=True)
)

In [54]:
users_df_pt2

,post_author,post_count
0,schedulerplus,67
1,AutoModerator,32


In [55]:
users_df = (
    users_df_pt1
    .rename(columns={"author": "username"})
    .merge(
        users_df_pt2.rename(columns={"post_author": "username"}),
        on="username",
        how="outer",
    )
    .fillna({
        "comment_count": 0,
        "post_count": 0,
    })
)

users_df[["comment_count", "post_count"]] = (
    users_df[["comment_count", "post_count"]].astype(int)
)

In [56]:
users_df

,username,comment_count,post_count
0,-Afya-,1,0
1,-AlmondButter-,1,0
2,-Blixx-,1,0
3,-Calypso,3,0
4,-Carbon-,3,0
...,...,...,...
14353,zxcasdqwd,2,0
14354,zz657,1,0
14355,zzeep21,2,0
14356,zzzzzaaaaaa130,4,0


In [62]:
users_df = users_df.rename(columns={
    "username": "author",
    "comment_count": "num_comments",
    "post_count": "num_submissions",
})

users_df["is_bot"] = (
    users_df["author"]
    .str.contains("automoderator", case=False, na=False)
    .map({True: "Y", False: "N"})
)

In [63]:
users_df.head(3)

,author,num_comments,num_submissions,is_bot
0,-Afya-,1,0,N
1,-AlmondButter-,1,0,N
2,-Blixx-,1,0,N


Save to the duckdb.